In [13]:
# ============================================================================
# SECTION 1: Install Dependencies
# ============================================================================
# Install required packages
!pip install -q transformers==4.48.0 datasets==2.14.5 jiwer soundfile librosa evaluate accelerate

In [3]:
# ============================================================================
# SECTION 2: Import Libraries
# ============================================================================
import os
import json
import pandas as pd
import numpy as np
import librosa
import soundfile as sf
from pathlib import Path
from typing import Dict, List, Any, Optional
import torch
from datasets import Dataset, DatasetDict, Audio, load_metric
from transformers import (
    Wav2Vec2CTCTokenizer,
    Wav2Vec2FeatureExtractor,
    Wav2Vec2Processor,
    Wav2Vec2ForCTC,
    TrainingArguments,
    Trainer
)
from dataclasses import dataclass
import evaluate
# Check GPU
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

2026-01-03 02:35:53.863202: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1767407754.051445      47 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1767407754.104874      47 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

PyTorch version: 2.6.0+cu124
CUDA available: True
GPU: Tesla T4


In [14]:
# ============================================================================
# SECTION 3: Dataset Preparation
# ============================================================================
# Path configuration
DATASET_PATH = "/kaggle/input/large-sinhala-asr-training-dataset"
OUTPUT_PATH = "/kaggle/working/sinhala-asr-model"
# Load train and test CSV files
train_df = pd.read_csv(f"{DATASET_PATH}/train.csv")
test_df = pd.read_csv(f"{DATASET_PATH}/test.csv")
print(f"Training samples: {len(train_df)}")
print(f"Test samples: {len(test_df)}")
print("\nSample data:")
print(train_df.head())
# Verify column names and adjust if needed
# Expected columns: 'path' or 'audio' and 'sentence' or 'transcription'
# Adjust these based on your actual CSV structure
AUDIO_COL = 'file'  # Change to your audio column name
TEXT_COL = 'sentence'  # Change to your text column name

Training samples: 132574
Test samples: 23396

Sample data:
   Unnamed: 0    filename      x  \
0       69861  70d725d61b  6e92f   
1       77001  7f2e316d14  3b15d   
2      108545  bc470065df  4b5d0   
3       47802  4b48bbffc5  8e991   
4       28441  2f2951d11c  936a6   

                                            sentence             full  \
0           ශ්‍රාවක චරිත නිදසුන් කොට පැහැදිලි කරන්න.  70d725d61b6e92f   
1                                      වෙන්න පුළුවනි  7f2e316d143b15d   
2                                     එය තමයි ඔවුන්ට  bc470065df4b5d0   
3                       සප්ත ආර්ය ධනයෙහි එක් කොටසකි.  4b48bbffc58e991   
4  මුදලාලි නිවසේ දොර විවෘත කිරීමෙන් පසු එය වසා නො...  2f2951d11c936a6   

                                  file  
0  asr_sinhala/data/70/70d725d61b.flac  
1  asr_sinhala/data/7f/7f2e316d14.flac  
2  asr_sinhala/data/bc/bc470065df.flac  
3  asr_sinhala/data/4b/4b48bbffc5.flac  
4  asr_sinhala/data/2f/2f2951d11c.flac  


In [15]:
# ============================================================================
# SECTION 4: Create HuggingFace Dataset
# ============================================================================
def prepare_dataset(df, dataset_path):
    """Convert pandas DataFrame to HuggingFace Dataset format"""
    data = {
        "audio": [],
        "transcription": []
    }
    
    for idx, row in df.iterrows():
        # Construct full audio path
        audio_file = os.path.join(dataset_path, row[AUDIO_COL])
        
        if os.path.exists(audio_file):
            data["audio"].append(audio_file)
            data["transcription"].append(row[TEXT_COL])
        else:
            print(f"Warning: Audio file not found: {audio_file}")
    
    return Dataset.from_dict(data)
# Create datasets
train_dataset = prepare_dataset(train_df, DATASET_PATH)
test_dataset = prepare_dataset(test_df, DATASET_PATH)
# Cast audio column to Audio type (this will handle loading)
train_dataset = train_dataset.cast_column("audio", Audio(sampling_rate=16000))
test_dataset = test_dataset.cast_column("audio", Audio(sampling_rate=16000))
print(f"\nProcessed datasets:")
print(f"Train: {len(train_dataset)} samples")
print(f"Test: {len(test_dataset)} samples")


Processed datasets:
Train: 132574 samples
Test: 23396 samples


In [16]:
# ============================================================================
# SECTION 5: Create Vocabulary (Sinhala Unicode Characters)
# ============================================================================
def extract_all_chars(batch):
    """Extract all unique characters from transcriptions"""
    all_text = " ".join(batch["transcription"])
    vocab = list(set(all_text))
    return {"vocab": [vocab], "all_text": [all_text]}
# Extract vocabulary from training data
vocab_train = train_dataset.map(
    extract_all_chars,
    batched=True,
    batch_size=-1,
    keep_in_memory=True,
    remove_columns=train_dataset.column_names
)
vocab_test = test_dataset.map(
    extract_all_chars,
    batched=True,
    batch_size=-1,
    keep_in_memory=True,
    remove_columns=test_dataset.column_names
)
# Combine vocabularies
vocab_list = list(set(vocab_train["vocab"][0]) | set(vocab_test["vocab"][0]))
vocab_dict = {v: k for k, v in enumerate(sorted(vocab_list))}
# Add special tokens
vocab_dict["|"] = vocab_dict[" "]  # Word delimiter
del vocab_dict[" "]
vocab_dict["[UNK]"] = len(vocab_dict)
vocab_dict["[PAD]"] = len(vocab_dict)
print(f"\nVocabulary size: {len(vocab_dict)}")
print(f"Sample characters: {list(vocab_dict.keys())[:20]}")
# Save vocabulary
os.makedirs(OUTPUT_PATH, exist_ok=True)
with open(f"{OUTPUT_PATH}/vocab.json", "w", encoding="utf-8") as f:
    json.dump(vocab_dict, f, ensure_ascii=False, indent=2)

Map:   0%|          | 0/132574 [00:00<?, ? examples/s]

Map:   0%|          | 0/23396 [00:00<?, ? examples/s]


Vocabulary size: 153
Sample characters: ['\t', '\n', '!', '"', '%', '&', "'", '+', ',', '.', '/', '0', '1', '2', '3', '4', '5', '6', '7', '8']


In [17]:
# ============================================================================
# SECTION 6: Initialize Tokenizer and Processor
# ============================================================================
# Create tokenizer from vocabulary
tokenizer = Wav2Vec2CTCTokenizer(
    f"{OUTPUT_PATH}/vocab.json",
    unk_token="[UNK]",
    pad_token="[PAD]",
    word_delimiter_token="|"
)
# Load feature extractor from pre-trained model
feature_extractor = Wav2Vec2FeatureExtractor(
    feature_size=1,
    sampling_rate=16000,
    padding_value=0.0,
    do_normalize=True,
    return_attention_mask=True
)
# Combine into processor
processor = Wav2Vec2Processor(
    feature_extractor=feature_extractor,
    tokenizer=tokenizer
)
print("\nTokenizer and processor initialized")
print(f"Vocab size: {len(processor.tokenizer)}")


Tokenizer and processor initialized
Vocab size: 155


In [18]:
# ============================================================================
# SECTION 7: Data Preprocessing
# ============================================================================
def prepare_dataset_for_training(batch):
    """Preprocess audio and text for training"""
    # Load and resample audio
    audio = batch["audio"]
    
    # Process audio
    batch["input_values"] = processor(
        audio["array"],
        sampling_rate=audio["sampling_rate"]
    ).input_values[0]
    
    batch["input_length"] = len(batch["input_values"])
    
    # Process text
    with processor.as_target_processor():
        batch["labels"] = processor(batch["transcription"]).input_ids
    
    return batch
# Apply preprocessing
train_dataset = train_dataset.map(
    prepare_dataset_for_training,
    remove_columns=train_dataset.column_names,
    num_proc=4
)
test_dataset = test_dataset.map(
    prepare_dataset_for_training,
    remove_columns=test_dataset.column_names,
    num_proc=4
)
print("\nDataset preprocessing complete")

Map (num_proc=4):   0%|          | 0/132574 [00:00<?, ? examples/s]

/usr/local/lib/python3.11/dist-packages/transformers/models/wav2vec2/processing_wav2vec2.py:174: UserWarning: `as_target_processor` is deprecated and will be removed in v5 of Transformers. You can process your labels by using the argument `text` of the regular `__call__` method (either in the same call as your audio inputs, or in a separate call.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/transformers/models/wav2vec2/processing_wav2vec2.py:174: UserWarning: `as_target_processor` is deprecated and will be removed in v5 of Transformers. You can process your labels by using the argument `text` of the regular `__call__` method (either in the same call as your audio inputs, or in a separate call.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/transformers/models/wav2vec2/processing_wav2vec2.py:174: UserWarning: `as_target_processor` is deprecated and will be removed in v5 of Transformers. You can process your labels by using the argument `text` of the regular `__call

RuntimeError: One of the subprocesses has abruptly died during map operation.To debug the error, disable multiprocessing.

In [ ]:
# ============================================================================
# SECTION 8: Data Collator (for Dynamic Padding)
# ============================================================================
@dataclass
class DataCollatorCTCWithPadding:
    """
    Data collator that will dynamically pad the inputs received.
    """
    processor: Wav2Vec2Processor
    padding: bool = True
    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        # Split inputs and labels
        input_features = [{"input_values": feature["input_values"]} for feature in features]
        label_features = [{"input_ids": feature["labels"]} for feature in features]
        batch = self.processor.pad(
            input_features,
            padding=self.padding,
            return_tensors="pt",
        )
        with self.processor.as_target_processor():
            labels_batch = self.processor.pad(
                label_features,
                padding=self.padding,
                return_tensors="pt",
            )
        # Replace padding with -100 to ignore in loss
        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )
        batch["labels"] = labels
        return batch
data_collator = DataCollatorCTCWithPadding(processor=processor, padding=True)

In [ ]:
# ============================================================================
# SECTION 9: Evaluation Metrics
# ============================================================================
wer_metric = evaluate.load("wer")
def compute_metrics(pred):
    """Compute Word Error Rate (WER)"""
    pred_logits = pred.predictions
    pred_ids = np.argmax(pred_logits, axis=-1)
    pred.label_ids[pred.label_ids == -100] = processor.tokenizer.pad_token_id
    pred_str = processor.batch_decode(pred_ids)
    label_str = processor.batch_decode(pred.label_ids, group_tokens=False)
    wer = wer_metric.compute(predictions=pred_str, references=label_str)
    return {"wer": wer}

In [ ]:
# ============================================================================
# SECTION 10: Load Pre-trained Model
# ============================================================================
model = Wav2Vec2ForCTC.from_pretrained(
    "facebook/wav2vec2-large-xlsr-53",
    attention_dropout=0.1,
    hidden_dropout=0.1,
    feat_proj_dropout=0.0,
    mask_time_prob=0.05,
    layerdrop=0.1,
    ctc_loss_reduction="mean",
    pad_token_id=processor.tokenizer.pad_token_id,
    vocab_size=len(processor.tokenizer),
    ctc_zero_infinity=True
)
# Freeze feature encoder (speeds up training)
model.freeze_feature_encoder()
print("\nModel loaded successfully")
print(f"Model parameters: {model.num_parameters() / 1e6:.2f}M")


In [ ]:
# ============================================================================
# SECTION 11: Training Configuration
# ============================================================================
training_args = TrainingArguments(
    output_dir=OUTPUT_PATH,
    group_by_length=True,
    per_device_train_batch_size=8,  # Adjust based on GPU memory
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    eval_strategy="steps",
    eval_steps=500,
    save_steps=500,
    logging_steps=100,
    learning_rate=3e-4,
    warmup_steps=500,
    num_train_epochs=15,  # Increase for better results
    save_total_limit=2,
    fp16=True,  # Mixed precision training
    push_to_hub=False,
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    report_to="none"
)

In [ ]:
# ============================================================================
# SECTION 12: Initialize Trainer
# ============================================================================
trainer = Trainer(
    model=model,
    data_collator=data_collator,
    args=training_args,
    compute_metrics=compute_metrics,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    tokenizer=processor.feature_extractor,
)
print("\nTrainer initialized")
print("Starting training...")

In [ ]:
# ============================================================================
# SECTION 13: Train Model
# ============================================================================
# Train the model
trainer.train()
print("\n✅ Training complete!")

In [ ]:
# ============================================================================
# SECTION 14: Evaluate Model
# ============================================================================
# Evaluate on test set
results = trainer.evaluate()
print("\n" + "="*50)
print("FINAL EVALUATION RESULTS")
print("="*50)
print(f"Word Error Rate (WER): {results['eval_wer']*100:.2f}%")
print("="*50)

In [ ]:
# ============================================================================
# SECTION 15: Save Model
# ============================================================================
# Save the final model
final_model_path = f"{OUTPUT_PATH}/final"
trainer.save_model(final_model_path)
processor.save_pretrained(final_model_path)
print(f"\n✅ Model saved to: {final_model_path}")

In [ ]:
# ============================================================================
# SECTION 16: Test Inference
# ============================================================================
# Test the model with a sample
def transcribe_audio(audio_path, model, processor):
    """Test transcription on a single audio file"""
    speech, sr = librosa.load(audio_path, sr=16000)
    
    input_values = processor(
        speech,
        sampling_rate=16000,
        return_tensors="pt"
    ).input_values
    
    with torch.no_grad():
        logits = model(input_values.to(model.device)).logits
    
    predicted_ids = torch.argmax(logits, dim=-1)
    transcription = processor.batch_decode(predicted_ids)[0]
    
    return transcription
# Test on first test sample
test_sample = test_df.iloc[0]
test_audio_path = os.path.join(DATASET_PATH, test_sample[AUDIO_COL])
test_transcription = test_sample[TEXT_COL]
if os.path.exists(test_audio_path):
    predicted = transcribe_audio(test_audio_path, model, processor)
    
    print("\n" + "="*50)
    print("SAMPLE TRANSCRIPTION TEST")
    print("="*50)
    print(f"Expected:  {test_transcription}")
    print(f"Predicted: {predicted}")
    print("="*50)


In [ ]:
# ============================================================================
# SECTION 17: Export Instructions
# ============================================================================
print("\n" + "="*70)
print("NEXT STEPS - DEPLOYING YOUR MODEL")
print("="*70)
print("\n1. Download the model folder:")
print(f"   - Location: {final_model_path}")
print("   - Files needed: config.json, model.safetensors, preprocessor_config.json,")
print("                   tokenizer_config.json, vocab.json")
print("\n2. Replace your backend model:")
print("   - Copy downloaded files to:")
print("     c:\\SLIIT\\sinhala_app(client+server)\\sinhala_clientside\\sinhala_app\\")
print("     sinhala_app_backend\\api\\models\\sinhala_asr\\")
print("\n3. Restart your backend server")
print("\n4. Test with real voice input")
print("="*70)
print("\n✅ ALL DONE! Your Sinhala ASR model is ready for deployment.")